# For MARIAN MT GPU- Installs, imports, constants

In [2]:
pip install pandas torch transformers datasets sacrebleu accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.1 MB/s eta 0:00:00


In [3]:
# Installs MT tooling
# Imports: core libs, HuggingFace seq2seq, datasets, and BLEU metrics
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import sacrebleu

## Extra tokenizer utilities sometimes needed by Marian

In [4]:
pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 9.4 MB/s eta 0:00:00


# Constants & device selection

In [5]:
# Model names, sequence limits, training budget, and device selection
# Sets which MarianMT checkpoints to use and defines core training hyperparameters.
# MAX_LENGTH controls the maximum token length per sentence; BATCH_SIZE and GRAD_ACCUM control the effective batch size.
# HING_EPOCHS and SPAN_EPOCHS set how many epochs each language model is fine-tuned for.
# get_device() picks CUDA or MPS when available (unless FORCE_CPU is True), and prints the final device used.


HING_MODEL = "Helsinki-NLP/opus-mt-hi-en"
SPAN_MODEL = "Helsinki-NLP/opus-mt-es-en"

MAX_LENGTH = 64
BATCH_SIZE = 4
GRAD_ACCUM = 4

HING_EPOCHS = 20
SPAN_EPOCHS = 10

FORCE_CPU = True


def get_device():
    if not FORCE_CPU and torch.cuda.is_available():
        return "cuda"
    if not FORCE_CPU and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = get_device()
print("Using device:", device)

Using device: cpu


# Train, validation and test sets

In [6]:
# Loads the prepared Hinglish and Spanglish datasets from CSV files into pandas DataFrames.
# Each language has separate train, validation, and test splits for supervised fine-tuning and evaluation.


print("Loading data...")
hing_train = pd.read_csv("/hinglish_train.csv")
hing_val = pd.read_csv("/hinglish_val.csv")
hing_test = pd.read_csv("/hinglish_test.csv")

span_train = pd.read_csv("/spanglish_train.csv")
span_val = pd.read_csv("/spanglish_val.csv")
span_test = pd.read_csv("/spanglish_test.csv")

print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")


Loading data...
Hinglish: 753 train, 94 val, 95 test
Spanglish: 847 train, 106 val, 106 test


# Tokenization helper

In [7]:
# Helper functions: tokenize data, compute BLEU/chrF, and eval on test
# tokenize_df converts a DataFrame of source/target text into tokenized tensors ready for seq2seq training.
# compute_metrics decodes model outputs and computes BLEU and chrF scores for validation.
# evaluate_model generates translations on the test set in batches, and returns predictions along with metrics.


def tokenize_df(df, tokenizer, prefix=""):
    src = [(prefix + t) for t in df["source"].astype(str)]
    tgt = df["target"].astype(str).tolist()

    enc = tokenizer(src, max_length=MAX_LENGTH, truncation=True, padding=False)
    dec = tokenizer(tgt, max_length=MAX_LENGTH, truncation=True, padding=False)

    return Dataset.from_dict(
        {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": dec["input_ids"]
        }
    )

def compute_metrics(eval_pred, tokenizer):
    preds, labels = eval_pred

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels]).score

    return {"bleu": bleu, "chrf": chrf}


def evaluate_model(path, test_df, tokenizer):
    model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
    model.eval()

    refs = test_df["target"].astype(str).tolist()
    preds = []

    batch = 32
    src = test_df["source"].astype(str).tolist()

    for i in range(0, len(src), batch):
        chunk = src[i:i+batch]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(device)

        with torch.no_grad():
            out = model.generate(**enc, max_length=MAX_LENGTH, num_beams=4)

        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

    preds = [p.strip() for p in preds]
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    em = 100.0 * sum(p.lower() == r.lower() for p, r in zip(preds, refs)) / len(refs)

    return preds, bleu, chrf, em

# HINGLISH training block - 20 EPOCHS

In [8]:
# Fine-tunes the Marian hi→en model on the Hinglish dataset for 20 epochs.
# It loads the tokenizer/model, tokenizes the train/val splits with a translation prefix,
# configures training hyperparameters (batch size, learning rate, eval/save each epoch),
# then uses Seq2SeqTrainer to run the training loop and finally saves the fine-tuned model + tokenizer.


print("TRAINING HINGLISH MODEL")


tokenizer_h = AutoTokenizer.from_pretrained(HING_MODEL)
model_h = AutoModelForSeq2SeqLM.from_pretrained(HING_MODEL).to(device)

train_ds = tokenize_df(hing_train, tokenizer_h, prefix="translate Hinglish to English: ")
val_ds   = tokenize_df(hing_val, tokenizer_h, prefix="translate Hinglish to English: ")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_h, model=model_h)

hing_args = Seq2SeqTrainingArguments(
    output_dir="models/marian_hinglish",
    num_train_epochs=HING_EPOCHS,
    learning_rate=1e-4,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",            # << fix here
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    no_cuda=(device != "cuda"),
)

hing_trainer = Seq2SeqTrainer(
    model=model_h,
    args=hing_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer_h,
    compute_metrics=lambda x: compute_metrics(x, tokenizer_h),
)

hing_trainer.train()
hing_trainer.save_model("models/marian_hinglish/best_model")
tokenizer_h.save_pretrained("models/marian_hinglish/best_model")


TRAINING HINGLISH MODEL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/304M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/tmp/ipython-input-2792398823.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  hing_trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,No log,3.183623,4.859766,23.690334
2,4.334800,2.806780,8.137317,27.244856
3,2.923900,2.592597,9.549296,30.654324
4,2.254400,2.517993,9.298016,33.016946
5,1.760900,2.458920,11.195923,32.860404
6,1.424700,2.429988,11.877454,35.397804
7,1.093500,2.432487,12.818285,35.404830
8,0.882700,2.461440,12.461526,34.672085
9,0.675300,2.472589,17.769416,38.765026
10,0.553900,2.474676,16.686842,37.493384


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[61126]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('models/marian_hinglish/best_model/tokenizer_config.json',
 'models/marian_hinglish/best_model/special_tokens_map.json',
 'models/marian_hinglish/best_model/vocab.json',
 'models/marian_hinglish/best_model/source.spm',
 'models/marian_hinglish/best_model/target.spm',
 'models/marian_hinglish/best_model/added_tokens.json')

# SPANGLISH training block - 10 EPOCHS

In [9]:
# Fine-tunes the Marian es→en model on the Spanglish dataset for SPAN_EPOCHS.
# It loads the Spanish→English Marian checkpoint, tokenizes Spanglish train/val splits,
# sets training hyperparameters (small LR, batch size, eval/save every epoch),
# then runs Seq2SeqTrainer to train the model and finally saves the fine-tuned weights and tokenizer.


print("TRAINING SPANGLISH MODEL")


tokenizer_s = AutoTokenizer.from_pretrained(SPAN_MODEL)
model_s = AutoModelForSeq2SeqLM.from_pretrained(SPAN_MODEL).to(device)

train_ds = tokenize_df(span_train, tokenizer_s)
val_ds = tokenize_df(span_val, tokenizer_s)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_s, model=model_s)

span_args = Seq2SeqTrainingArguments(
    output_dir="models/marian_spanglish",
    num_train_epochs=SPAN_EPOCHS,
    learning_rate=1e-5,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",           # << fix
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    no_cuda=(device != "cuda"),
)

span_trainer = Seq2SeqTrainer(
    model=model_s,
    args=span_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer_s,
    compute_metrics=lambda x: compute_metrics(x, tokenizer_s),
)

span_trainer.train()
span_trainer.save_model("models/marian_spanglish/best_model")
tokenizer_s.save_pretrained("models/marian_spanglish/best_model")






TRAINING SPANGLISH MODEL


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/tmp/ipython-input-1140431505.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  span_trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,2.998700,2.352787,55.823994,76.480149
2,2.152700,1.955164,52.591475,74.181728
3,1.858600,1.752614,51.749184,73.194064
4,1.667100,1.631091,50.441914,71.475799
5,1.513100,1.551930,49.363424,70.768680
6,1.429700,1.501246,49.136691,70.319658
7,1.367200,1.465289,49.477709,70.640948
8,1.307100,1.441570,49.754293,70.643052
9,1.286200,1.427855,49.400715,70.432260
10,1.258400,1.424307,49.487274,70.395672


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('models/marian_spanglish/best_model/tokenizer_config.json',
 'models/marian_spanglish/best_model/special_tokens_map.json',
 'models/marian_spanglish/best_model/vocab.json',
 'models/marian_spanglish/best_model/source.spm',
 'models/marian_spanglish/best_model/target.spm',
 'models/marian_spanglish/best_model/added_tokens.json')

# Evaluating model performance

In [10]:
# Evaluate both fine-tuned models on test sets and save CSV outputs

print("\n" + "="*70)
print("EVALUATING MODELS")
print("="*70)

print("\nHinglish Evaluation:")
hing_preds, hing_bleu, hing_chrf, hing_em = evaluate_model(
    "models/marian_hinglish/best_model", hing_test, tokenizer_h
)
print(f"BLEU={hing_bleu:.2f}, chrF={hing_chrf:.2f}, EM={hing_em:.2f}%")

print("\nSpanglish Evaluation:")
span_preds, span_bleu, span_chrf, span_em = evaluate_model(
    "models/marian_spanglish/best_model", span_test, tokenizer_s
)
print(f"BLEU={span_bleu:.2f}, chrF={span_chrf:.2f}, EM={span_em:.2f}%")

hing_test["marian_prediction"] = hing_preds
span_test["marian_prediction"] = span_preds

hing_test.to_csv("hinglish_marian_results.csv", index=False)
span_test.to_csv("spanglish_marian_results.csv", index=False)

print("\n Models trained and results saved.")


EVALUATING MODELS

Hinglish Evaluation:
BLEU=13.27, chrF=30.66, EM=6.32%

Spanglish Evaluation:
BLEU=48.12, chrF=67.16, EM=0.94%

 Models trained and results saved.


# Qualitative samples

In [11]:
# Utility to reload a model and print random qualitative examples

import random

def show_examples(model_path, tokenizer, test_df, num_examples=5):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    print("\n" + "="*60)
    print(f" Sample Translations ({model_path})")
    print("="*60)

    samples = test_df.sample(num_examples)

    for _, row in samples.iterrows():
        src = row["source"]
        tgt = row["target"]

        encoded = tokenizer(
            src,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)

        with torch.no_grad():
            output = model.generate(**encoded, max_length=MAX_LENGTH, num_beams=6)

        pred = tokenizer.decode(output[0], skip_special_tokens=True)

        print("\n Source:", src)
        print("Target (reference):", tgt)
        print("Model Prediction:", pred)


# Qualitative sample for - HINGLISH

In [12]:
# Show a few random Hinglish → English translations

show_examples(
    "models/marian_hinglish/best_model",
    tokenizer_h,
    hing_test,
    num_examples=5
)



 Sample Translations (models/marian_hinglish/best_model)

 Source: mujhe pata hai ki maine dekha hai lakin phir bhi wo questions poochne hai  :)
Target (reference): i know i've watched it but still need to ask those questions :)
Model Prediction: I think she was a very interestion. :)

 Source: mein maantha hoon. muje bhi dikh raha hoon ki yeh batman apni life ka crime ko fight karne ka pursuing kartha hein
Target (reference): I agree.  I can see why it led to Batman pursuing a life of crime fighting.
Model Prediction: I was also really diked to me, it also really happened to be pursuing their own life.

 Source: haan
Target (reference): yes
Model Prediction: Han

 Source: Mai tumse sehmat hu ispar. Mujhe wo itni buri bhi nahi lagi kyuki usne Nick aur Judy ko security camera ka access dia.
Target (reference): I have to agree with you on that. I don't see her as much of a bad guy, since she offered Nick and Judy access to the security cameras.
Model Prediction: I am not sure. I'm not s

# Qualitative sample for - SPANGLISH

In [13]:
# Show a few random Spanglish → English translations

show_examples(
    "models/marian_spanglish/best_model",
    tokenizer_s,
    span_test,
    num_examples=5
)


 Sample Translations (models/marian_spanglish/best_model)

 Source: Eso es really true. Lo said the other night, y lo repeat now: esto no es a political issue.
Target (reference): That's really true. I said the other night, and I'll repeat now: this is not a political issue.
Model Prediction: That's really true. Lo said the other night, and lo repeat now: this is not a polital issue.

 Source: Este language is comunicando that it está surprised to see you, y está interested in looking at you.
Target (reference): This language is communicating that it is surprised to see you, and it's interested in looking at you.
Model Prediction: This language is comunicing that it's surprised to see you, and it's interested in looking at you.

 Source: "Como había lost my jaw, ya no podía form a seal, y por tanto my tongue y todo el resto del vocal equipment había quedado powerless."
Target (reference): Because I had lost my jaw, I could no longer form a seal, and therefore my tongue and all of my o